**Putusan structured-extractor SFT (Stage 1)** — `unsloth/Qwen3.5-9B`, language-only. Reconstructs the model input from each decision's own section spans and finetunes it to emit the 31-section JSON (see [`RAG/ORCHESTRATION.md`](../RAG/ORCHESTRATION.md)). Designed for an **A100 80GB** (putusan contexts are long).
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### Unsloth

In [ ]:
from unsloth import FastLanguageModel  # language-only SFT (was FastVisionModel)
import torch

# Putusan bodies are long — the input is the FULL document even though each example now
# asks for only 1-2 sections. 12.8% of docs exceed 32k input tokens but only 3.7% exceed
# 48k, so we train at 48k; 4-bit QLoRA keeps that within the A100 40GB.
max_seq_length = 49152  # RoPE-scaled long context; covers ~96% of document inputs.

model, tokenizer = FastLanguageModel.from_pretrained(
    "Qwen/Qwen3.5-9B",              # Stage 1/2 base per RAG/ORCHESTRATION.md (was Qwen3.5-4B)
    max_seq_length = max_seq_length,
    load_in_4bit = True,              # 4-bit QLoRA: needed for 48k context on a 40GB A100
    load_in_8bit = False,
    full_finetuning = False,
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for long context
)

We now add LoRA adapters so we only train ~1% of parameters. This is a **language-only** extractor (the vision path is removed), so we target the attention + MLP projection modules directly.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,            # The larger, the higher the accuracy, but might overfit
    # Qwen3.5 is a hybrid: 3 of every 4 layers are linear attention whose projections
    # (in_proj_qkvz / in_proj_ba / out_proj) aren't named q/k/v/o_proj — an explicit
    # target_modules list would leave them frozen. Let Unsloth's filters pick instead.
    finetune_vision_layers     = False,  # language-only extractor, keep the vision tower frozen
    finetune_language_layers   = True,
    finetune_attention_modules = True,   # covers full-attn q/k/v/o AND linear-attn projections
    finetune_mlp_modules       = True,   # gate/up/down_proj in all layers
    lora_alpha = 32,   # Recommended alpha == r at least
    lora_dropout = 0,  # Supports any, but = 0 is optimized
    bias = "none",     # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

# Verify adapter coverage: expect linear-attn projections here, not just q_proj..down_proj.
from collections import Counter
print(Counter(name.split(".")[-3] for name, _ in model.named_parameters() if "lora_A" in name))

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Counter({'lora_A': 128})


<a name="Data"></a>
### Data Prep — per-section extraction (`sft_sections`)

We fine-tune a **structured section extractor**: given a **full** putusan (court decision) body, emit one JSON object containing only the **1-2 requested sections** — matching inference, where a user uploads a whole decision and asks for one or two sections.

The dataset lives in [`Haeryz/putusan-structured-extraction`](https://huggingface.co/datasets/Haeryz/putusan-structured-extraction), config **`sft_sections`** (built by `notebooks/build_sections_dataset.py` from the legacy whole-doc `sft` config — same document-disjoint splits, zero leakage). Each document yields six examples with deterministic, difficulty-weighted section sampling: `ahli` (67% empty in gold) every doc, `penangkapan`/`surat` alternating, two of the five long-body sections rotating, one medium section rotating, and one rotating pair of trivial identity/date sections. ~16% of examples ask for a section that is empty in that document, so the model learns to answer with an empty list instead of hallucinating.

Each row carries a ready `messages` column (system prompt naming the requested sections, full putusan body as `user`, gold per-section JSON as `assistant`), so we just load the config below — no local build step needed.

In [ ]:
# Load the per-section SFT config from the Hub (configs: sft / sft_sections / grpo / rag).
from datasets import load_dataset

DATASET_REPO = "Haeryz/putusan-structured-extraction"
dataset      = load_dataset(DATASET_REPO, "sft_sections", split = "train")
eval_dataset = load_dataset(DATASET_REPO, "sft_sections", split = "validation")

Let's look at the dataset — each row is a chat with a `system` extraction instruction, the reconstructed putusan body as `user`, and the gold 31-section JSON as `assistant`.

In [6]:
dataset

Dataset({
    features: ['id', 'corpus', 'annotator_model', 'source_file', 'source_sha256', 'extraction_method', 'purpose', 'split', 'split_seed', 'input_text', 'target_json', 'sections_json', 'messages', 'prompt', 'answer', 'n_sections', 'n_nonempty_sections', 'empty_sections', 'n_empty_sections', 'cross_model_fill_json', 'n_sections_filled_cross_model', 'models_covering_doc', 'n_input_chars', 'n_input_words', 'n_target_chars'],
    num_rows: 2468
})

In [7]:
# Peek at the assistant target (gold JSON) for the first example.
print(dataset[0]["messages"][2]["content"][:1000])

{"status": "completed", "source_file": "10_Pid.Sus-Anak_2021_PN_Unh.txt", "source_sha256": "4ed5f4fe0d6fdfdc4c52dadfa794f4d95ed60d4dea000afd69a87c8e6e239688", "sections": {"judul": ["P U T U S A N"], "nomor_putusan": ["Nomor Disamarkan/Pid.Sus-Anak/2021/PN Unh"], "irah_irah": ["DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA"], "nama_pengadilan_negeri": ["Pengadilan Negeri Unaaha yang mengadili perkara pidana anak dengan\nacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai\nberikut dalam perkara Anak:"], "keterangan_perkara": ["Pengadilan Negeri Unaaha yang mengadili perkara pidana anak dengan\nacara pemeriksaan biasa dalam tingkat pertama menjatuhkan putusan sebagai\nberikut dalam perkara Anak:"], "nama_lengkap": ["Anak"], "tempat_lahir": ["Unaaha"], "umur_tanggal_lahir": ["15 Tahun / 07 Maret 2006"], "jenis_kelamin": ["Laki-laki"], "kebangsaan": ["Indonesia"], "tempat_tinggal": ["Desa Lalongowuna Kec. Tongauna Kab. Konawe"], "agama": ["Islam"], "pekerjaan": ["

We format each chat into a single `text` string with the model's chat template, so the standard text `SFTTrainer` can tokenize it (no vision collator needed).

In [8]:
def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(msgs, tokenize = False, add_generation_prompt = False)
        for msgs in examples["messages"]
    ]
    return {"text": texts}

dataset      = dataset.map(formatting_prompts_func, batched = True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/2468 [00:00<?, ? examples/s]

Map:   0%|          | 0/311 [00:00<?, ? examples/s]

Measure the tokenized length distribution and set `max_length` from the **90th percentile** (per ORCHESTRATION Stage 1), capped at the model's `max_seq_length`.

In [ ]:
import numpy as np, os, wandb
from tqdm.auto import tqdm

# Qwen3.5 is a vision-language model, so `tokenizer` is actually a Processor whose first
# positional arg is `images` — calling it on a raw string routes text into load_image().
# Use the underlying text tokenizer for pure-text token counting.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# Measuring 14.8k long documents takes minutes — cache the lengths as a W&B Artifact
# so reruns skip straight to the percentiles. Requires being logged in to wandb here
# (the SFTTrainer below reuses this same run via report_to="wandb").
ARTIFACT = "token-lengths-sft_sections-train"
run = wandb.run or wandb.init(project = "huggingface")

token_lengths = None
try:
    art_dir = run.use_artifact(f"{ARTIFACT}:latest").download()
    cached = np.load(os.path.join(art_dir, "token_lengths.npy"))
    if len(cached) == len(dataset):  # invalidate the cache if the dataset changed
        token_lengths = cached
        print(f"Loaded {len(cached)} cached token lengths from W&B artifact {ARTIFACT}:latest")
except Exception as e:
    print(f"No usable cached measurement ({type(e).__name__}) — measuring now")

if token_lengths is None:
    # Batched tokenization: the fast tokenizer parallelizes across a batch,
    # far quicker than calling it once per document.
    texts = dataset["text"]
    token_lengths = np.array([
        len(ids)
        for i in tqdm(range(0, len(texts), 256), desc = "measuring token lengths", unit = "batch")
        for ids in text_tokenizer(texts[i : i + 256], add_special_tokens = False)["input_ids"]
    ])
    np.save("token_lengths.npy", token_lengths)
    art = wandb.Artifact(ARTIFACT, type = "measurement",
                         metadata = {"rows": len(dataset), "dataset": DATASET_REPO, "config": "sft_sections"})
    art.add_file("token_lengths.npy")
    run.log_artifact(art)
    print(f"Measured and cached {len(token_lengths)} token lengths to W&B artifact {ARTIFACT}")

p50, p90, p95 = (int(np.percentile(token_lengths, p)) for p in (50, 90, 95))
# Round p90 up to a multiple of 256; never exceed the model's context window.
MAX_LENGTH = int(min(np.ceil(p90 / 256) * 256, max_seq_length))
print(f"token length  p50={p50}  p90={p90}  p95={p95}  max={int(max(token_lengths))}")
print(f"MAX_LENGTH (90th pct, capped at {max_seq_length}) = {MAX_LENGTH}")

Here is the fully-formatted `text` for the first example (system + user putusan body + assistant JSON):

In [11]:
print(dataset[0]["text"][:2000])

<|im_start|>system
Anda adalah pengekstrak terstruktur putusan pengadilan Indonesia. Diberikan badan teks putusan, keluarkan SATU objek JSON dengan tepat 31 kunci bagian (dalam urutan kanonik). Setiap nilai adalah daftar kutipan verbatim (extractive) yang disalin persis dari teks sumber — jangan pernah memparafrasekan, meringkas, atau mengarang. Jika sebuah bagian tidak ada, gunakan daftar kosong dan cantumkan kuncinya di 'empty_sections'. Kunci bagian, dalam urutan: judul, nomor_putusan, irah_irah, nama_pengadilan_negeri, keterangan_perkara, nama_lengkap, tempat_lahir, umur_tanggal_lahir, jenis_kelamin, kebangsaan, tempat_tinggal, agama, pekerjaan, penangkapan, penahanan, tuntutan, dakwaan, saksi, ahli, terdakwa, surat, petunjuk_barang_bukti, fakta_hukum, pertimbangan_hukum, amar_putusan, hari, tanggal, tahun, siapa_yang_memutus, panitera_pengganti, tanda_tangan_majelis.<|im_end|>
<|im_start|>user
P U T U S A N

Nomor Disamarkan/Pid.Sus-Anak/2021/PN Unh

DEMI KEADILAN BERDASARKAN KETU

Before finetuning, let's see what the base model emits for the first putusan body (system + user only, assistant left blank).

In [13]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# Use the inner text tokenizer: the Qwen3VLProcessor's apply_chat_template defaults to
# tokenize=False and returns a str (ignoring return_tensors), so .to("cuda") would fail.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# system + user (drop the gold assistant turn); ask the model to produce the JSON.
prompt_messages = dataset[0]["messages"][:2]
inputs = text_tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(text_tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 512,
                   use_cache = True, temperature = 0.7, min_p = 0.1)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The user wants me to extract specific information from a provided Indonesian court judgment text and format it into a JSON object with exactly 31 keys in a canonical order.
The keys are:
1. judul
2. nomor_putusan
3. irah_irah
4. nama_pengadilan_negeri
5. keterangan_perkara
6. nama_lengkap
7. tempat_lahir
8. umur_tanggal_lahir
9. jenis_kelamin
10. kebangsaan
11. tempat_tinggal
12. agama
13. pekerjaan
14. penangkapan
15. penahanan
16. tuntutan
17. dakwaan
18. saksi
19. ahli
20. terdakwa
21. surat
22. petunjuk_barang_bukti
23. fakta_hukum
24. pertimbangan_hukum
25. amar_putusan
26. hari
27. tanggal
28. tahun
29. siapa_yang_memutus
30. panitera_pengganti
31. tanda_tangan_majelis

I need to scan the text for each of these sections and extract verbatim quotes. If a section is missing, I should use an empty list and note it in 'empty_sections'.

Let's analyze the text to find the content for each key.

1.  **judul**: "P U T U S A N"
2.  **nomor_putusan**: "Nomor Disamarkan/Pid.Sus-Anak/2021/P

<a name="Train"></a>
### Train the model

Standard text `SFTTrainer` (no `UnslothVisionDataCollator`). We train on the `text` field for `num_train_epochs = 2` (ORCHESTRATION Stage 1 says 1-3), with `max_length` set to the measured 90th-percentile token length. We then use `train_on_responses_only` so the loss is computed **only on the assistant JSON**, not on the (very long) putusan input. Set `max_steps` for a quick smoke test.

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)  # Enable for training!

# Qwen3.5-9B ships a Qwen3VLProcessor (verified in the model's preprocessor_config.json),
# so `tokenizer` is a multimodal processor whose __call__ takes `images` first and chokes
# on raw text. Hand the trainer the inner text tokenizer for this language-only SFT.
text_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

trainer = SFTTrainer(
    model = model,
    tokenizer = text_tokenizer,
    train_dataset = dataset,
    eval_dataset = eval_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,   # long sequences - keep the micro-batch small
        gradient_accumulation_steps = 8,   # effective batch size = 16
        warmup_steps = 5,
        num_train_epochs = 1,              # 14,766 per-section examples -> ~923 optimizer steps
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",     # For Weights and Biases
        max_length = MAX_LENGTH,
        eval_strategy = "steps",           # evaluate on the val split during training
        eval_steps = 200,                  # val is 1,866 long examples - per-step eval would dominate runtime
        per_device_eval_batch_size = 1,
    ),
)

wandb : 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

In [15]:
# Train only on the assistant response (the JSON), masking the putusan input.
# Qwen3.5 uses ChatML markers.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

Map:   0%|          | 0/2468 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2468 [00:00<?, ? examples/s]

Unsloth: Removed 342 out of 2468 samples from train_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


Map:   0%|          | 0/311 [00:00<?, ? examples/s]

Filter:   0%|          | 0/311 [00:00<?, ? examples/s]

Unsloth: Removed 26 out of 311 samples from eval_dataset where all labels were -100 (no response marker found, usually truncation). This prevents NaN loss during training.


In [16]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
28.227 GB of memory reserved.


In [17]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,126 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 58,195,968 of 9,468,009,712 (0.61% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: haeriz42069 (haeriz42069-universitas-muhammadiyah-malang) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.46 GiB. GPU 0 has a total capacity of 39.49 GiB of which 649.44 MiB is free. Including non-PyTorch memory, this process has 38.80 GiB memory in use. Of the allocated memory 34.55 GiB is allocated by PyTorch, and 1.60 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Run the finetuned extractor on a held-out putusan body - it should emit the 31-section JSON. We use low-temperature decoding (`temperature = 0.3`, `min_p = 0.1`) for stable structured output.

In [ ]:
FastLanguageModel.for_inference(model)  # Enable for inference!

# Try a held-out test-split example from the Hub repo (per-section config).
test_dataset = load_dataset(DATASET_REPO, "sft_sections", split = "test")
prompt_messages = test_dataset[0]["messages"][:2]  # system + user, drop the gold assistant turn

inputs = tokenizer.apply_chat_template(
    prompt_messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 2048,
                   use_cache = True, temperature = 0.3, min_p = 0.1)

<a name="Save"></a>
### Saving, loading finetuned models
Save the Stage-1 LoRA adapters as `qwen_extractor_sft_lora` - Stage 2 (GRPO) continues from this. Use `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, not the full model. To save 16-bit merged for serving, scroll down.

In [ ]:
model.save_pretrained("qwen_extractor_sft_lora")  # Local saving (Stage 2 continues from this)
tokenizer.save_pretrained("qwen_extractor_sft_lora")
# model.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_extractor_sft_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_extractor_sft_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        load_in_4bit = False, # Set to False for 16bit LoRA
    )
    FastLanguageModel.for_inference(model) # Enable for inference!

    prompt_messages = dataset[0]["messages"][:2]
    inputs = tokenizer.apply_chat_template(
        prompt_messages, add_generation_prompt = True, return_tensors = "pt",
    ).to("cuda")
    from transformers import TextStreamer
    text_streamer = TextStreamer(tokenizer, skip_prompt = True)
    _ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 2048,
                       use_cache = True, temperature = 0.3, min_p = 0.1)

### Saving to float16 for vLLM

We also support saving to `float16` directly for serving (Stage 3). Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit merged (serving-ready extractor)
if False: model.save_pretrained_merged("qwen_extractor_sft_merged", tokenizer,)

# To export and save to your Hugging Face account
if False: model.push_to_hub_merged("YOUR_USERNAME/qwen_extractor_sft_merged", tokenizer, token = "YOUR_HF_TOKEN")